# 9. Bias in the data: Titanic survival

BDI's `ebdai` package looks at **bias in the labels** and, after a fuzzy rule model is fitted, **bias in the rules that fire**. This notebook uses the Titanic table from the [WorkshopIgualdad2025](https://github.com/rferper/WorkshopIgualdad2025) workshop (Raquel Fernandez Peralta and Javier Fumanal Idocin), originally written for Ex-Fuzzy 2.1.3. The current `ex_fuzzy` API still trains the classifier; the new helpers live in `ebdai`.

`import ex_fuzzy` is the rule learner. `import ebdai` is the bias layer.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

from ex_fuzzy import BaseFuzzyRulesClassifier, FUZZY_SETS
from ex_fuzzy import eval_tools

from ebdai import (
    features_and_target,
    load_titanic,
    outcome_rates_by_group,
    plot_outcome_rates,
    plot_winning_rules_by_group,
    fairness_report,
    parse_printed_rules,
    winning_rules_by_group,
)

frame, sensitive = load_titanic()
X, y = features_and_target(frame, 'Survived')
print(X.head())
print('rows', len(X), 'sensitive', sensitive)

   Class     Sex   Age  SibSp  Parch     Fare Embarked
0      3    male  22.0      1      0   7.2500        S
1      1  female  38.0      1      0  71.2833        C
2      3  female  26.0      0      0   7.9250        S
3      1  female  35.0      1      0  53.1000        S
4      3    male  35.0      0      0   8.0500        S
rows 712 sensitive Sex


## Positive outcome by sex

Women survived at a much higher rate than men. That is **bias in the data**: the label `Survived` is associated with `Sex` before any model is trained.

In [2]:
rates = outcome_rates_by_group(y, X[sensitive], positive_label=1)
print(rates)
plot_outcome_rates(rates, title='Titanic survival rate by sex')

    group    n  n_positive  positive_rate
0  female  259         195       0.752896
1    male  453          93       0.205298


<Axes: title={'center': 'Titanic survival rate by sex'}, xlabel='Group', ylabel='Positive rate'>

## A small fuzzy rule classifier

The search is kept short so the notebook stays a demo. Current `BaseFuzzyRulesClassifier` still accepts `nRules`, `nAnts`, `n_linguistic_variables`, `ds_mode` and `categorical_mask` as in 2.1.3; missing values are rejected, which is why the loader dropped incomplete rows.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)
clf = BaseFuzzyRulesClassifier(
    nRules=8, nAnts=3, fuzzy_type=FUZZY_SETS.t1,
    n_linguistic_variables=3, ds_mode=1, verbose=False,
    n_gen=6, pop_size=12, patience=3, random_state=42,
)
clf.fit(X_train, y_train)
report = eval_tools.eval_fuzzy_model(
    clf, X_train, y_train, X_test, y_test,
    plot_rules=False, print_rules=True, plot_partitions=False,
    return_rules=True, bootstrap_results_print=False,
)
print(report[:1500] if report else '(no rule text)')

------------
ACCURACY
Train performance: 0.7526205450733753
Test performance: 0.6510638297872341
------------
MATTHEW CORRCOEF
Train performance: 0.4778647211133203
Test performance: 0.26215313062999657
------------
Rules for consequent: 0
----------------
IF Age IS Low AND Parch IS Medium AND Fare IS Medium WITH DS 0.000193219275412515, ACC 0.7853658536585366, WGHT 1.0
IF Sex IS female AND SibSp IS High WITH DS 0.0066384034664147195, ACC 0.8, WGHT 1.0
IF Parch IS Medium AND Fare IS Low AND Embarked IS C WITH DS 0.0001595028564434811, ACC 0.5, WGHT 1.0
IF Age IS Low AND SibSp IS Low WITH DS 0.04170820421016782, ACC 0.6506849315068494, WGHT 1.0
IF Parch IS High AND Fare IS Low WITH DS 0.006708595387840671, ACC 0.8, WGHT 1.0

Rules for consequent: 1
----------------
IF Sex IS female AND Age IS Medium WITH DS 0.10796843507156857, ACC 0.8256880733944955, WGHT 1.0


Rules for consequent: 0
----------------
IF Age IS Low AND Parch IS Medium AND Fare IS Medium WITH DS 0.000193219275412515, AC

## Bias in inference: which rules fire for whom

`explainable_predict` still returns winning-rule indexes. `winning_rules_by_group` counts them per sex.

In [4]:
y_pred = clf.predict(X_test)
table, gaps = fairness_report(y_test, y_pred, X_test[sensitive])
print(table)
print(pd.Series(gaps))

texts = parse_printed_rules(report or '')
counts = winning_rules_by_group(clf, X_test, X_test[sensitive], rule_texts=texts)
print(counts.head())
plot_winning_rules_by_group(counts, title='Winning Titanic rules by sex')

    group    n  selection_rate       tpr       fpr       fnr       tnr
0  female   86        0.569767  0.515625  0.727273  0.484375  0.272727
1    male  149        0.000000  0.000000  0.000000  1.000000  1.000000
demographic_parity_difference    0.569767
demographic_parity_ratio         0.000000
equalized_odds_difference        0.727273
equalized_odds_ratio             0.000000
tpr_difference                   0.515625
fpr_difference                   0.727273
dtype: float64
    group  rule  count      rate  \
0  female     0      9  0.104651   
1  female     1      1  0.011628   
2  female     2      2  0.023256   
3  female     3     24  0.279070   
4  female     4      1  0.011628   

                                           rule_text  
0  IF Age IS Low AND Parch IS Medium AND Fare IS ...  
1          IF Sex IS female AND SibSp IS High THEN 0  
2  IF Parch IS Medium AND Fare IS Low AND Embarke...  
3              IF Age IS Low AND SibSp IS Low THEN 0  
4            IF Parch IS Hig

<Axes: title={'center': 'Winning Titanic rules by sex'}, xlabel='Winning rule', ylabel='Share of group'>